# Lesson 8.3: What Is Multi-Query Retrieval and When Do You Use It?

**Companion notebook for Lesson 8.3**

---

| Section | What you will build |
|---|---|
| 1. Under-specification | Embed 4 interpretations of one query; see them land in different places |
| 2. ML Corpus | 12-doc corpus across 4 topic clusters; baseline failures |
| 3. Multi-query Fan-out | `MockMultiQueryLLM` + `multi_retrieve()` pipeline |
| 4. Deduplication | Simple set dedup vs. MMR (Maximal Marginal Relevance) |
| 5. Visualisations | PCA of corpus clusters + query angle coverage |
| 6. When It Hurts | Drift demo, latency/cost table |
| 7. Framework Code | LangChain + LlamaIndex patterns |
| 8. The Agentic Connection | Fan-out / fan-in as the shape of agentic retrieval |
| 9. Coverage@k Eval | Measure how many relevant clusters each strategy finds |
| 10. Claude API | Real LLM query generation |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`, `scikit-learn`  
**Optional (Section 10):** `anthropic`

In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib scikit-learn
# !pip install anthropic   # optional — Section 10

In [ ]:
%matplotlib inline

import os
import re
import warnings
from collections import defaultdict

os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

try:
    from sklearn.decomposition import PCA
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print('sklearn not installed. PCA visualisation in Section 5 will be skipped.')

print('Imports ready.')

---
## 1. The Under-Specification Problem

When a user asks "How do I make my model faster?", they could mean any of four very different things:

```
┌──────────────────────────────────────────────────────────────────────┐
│            ONE QUERY — FOUR DIFFERENT MEANINGS                       │
└──────────────────────────────────────────────────────────────────────┘

  "How do I make my model faster?"
          │
          ├─── Faster training?   → mixed precision, multi-GPU, gradient accumulation
          ├─── Faster inference?  → quantisation, distillation, dynamic batching
          ├─── Faster data load?  → workers, prefetch, caching
          └─── Faster converge?   → learning rate schedules, optimisers

  Each meaning lives in a DIFFERENT region of vector space.
  A single query embedding lands in one region — and retrieves from only that one.
  The other three regions are invisible.

  Multi-query fix: generate one query per interpretation, retrieve from all four.
```

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedder ready.')

# The original under-specified query
original_query = 'How do I make my model faster?'

# Four angle queries — one per interpretation
angle_queries = [
    'What GPU training optimizations reduce time-per-epoch?',
    'What techniques reduce model inference latency at serving time?',
    'How do I eliminate data loading bottlenecks during model training?',
    'What optimizer and learning rate settings speed up model convergence?',
]

# Embed all queries
all_queries = [original_query] + angle_queries
all_embs    = embedder.encode(all_queries, convert_to_tensor=True, show_progress_bar=False)

# Pairwise similarity between angle queries and original
print(f'Original query: {original_query}')
print()
print(f'{"Angle query":<58} {"Sim to original":<18} {"Sim to each other"}')
print('-' * 120)

for i, aq in enumerate(angle_queries):
    sim_to_orig = float(util.cos_sim(all_embs[0], all_embs[i + 1]))
    # Similarity to other angle queries (average)
    sims_to_others = [
        float(util.cos_sim(all_embs[i + 1], all_embs[j + 1]))
        for j in range(len(angle_queries)) if j != i
    ]
    avg_cross_sim = np.mean(sims_to_others)
    print(f'{aq:<58} {sim_to_orig:<18.3f} {avg_cross_sim:.3f} (avg cross-sim)')

print()
print('Key insight: the 4 angle queries have low cross-similarity — they embed to')
print('different regions of vector space, covering different neighbourhoods of the corpus.')

---
## 2. ML Optimisation Corpus + Baseline Retriever

12 documents spanning 4 ML speed topics. Each topic cluster lives in a different neighbourhood of embedding space — a single under-specified query will only land near one cluster.

In [ ]:
CORPUS = [
    # ── Faster training ───────────────────────────────────────────────────────
    {'id': 0,  'topic': 'Training speed',   'cluster': 0,
     'title': 'Mixed Precision Training with AMP',
     'text':  'Mixed precision training using NVIDIA AMP reduces GPU memory consumption '
              'and increases throughput by performing matrix multiplications in float16 '
              'while keeping master weights in float32. Typical speedups are 2-3x on '
              'Volta and Ampere architectures.'},

    {'id': 1,  'topic': 'Training speed',   'cluster': 0,
     'title': 'Gradient Accumulation for Large Batch Training',
     'text':  'Gradient accumulation simulates large batch sizes by accumulating gradients '
              'over multiple forward passes before performing a weight update. This allows '
              'effective batch sizes beyond GPU VRAM limits without sacrificing convergence.'},

    {'id': 2,  'topic': 'Training speed',   'cluster': 0,
     'title': 'Multi-GPU Training with DistributedDataParallel',
     'text':  'PyTorch DistributedDataParallel synchronises gradients across multiple GPUs '
              'using an all-reduce operation after each backward pass. Linear scaling of '
              'training throughput is achievable up to 8 GPUs with proper learning rate '
              'scaling.'},

    # ── Faster inference ──────────────────────────────────────────────────────
    {'id': 3,  'topic': 'Inference speed',  'cluster': 1,
     'title': 'INT8 Quantisation for Inference Acceleration',
     'text':  'Post-training quantisation converts model weights and activations from '
              'float32 to int8, reducing model size by 4x and improving inference '
              'latency by 2-4x on compatible hardware. Accuracy loss is typically under '
              '1% with calibration data.'},

    {'id': 4,  'topic': 'Inference speed',  'cluster': 1,
     'title': 'Knowledge Distillation for Model Compression',
     'text':  'Knowledge distillation transfers representations from a large teacher '
              'model to a smaller student model using soft targets. The student achieves '
              'near-teacher accuracy at a fraction of the parameters, enabling '
              'significantly faster inference at serving time.'},

    {'id': 5,  'topic': 'Inference speed',  'cluster': 1,
     'title': 'Dynamic Batching for Inference Throughput',
     'text':  'Dynamic batching aggregates concurrent inference requests into a single '
              'batched forward pass, maximising GPU utilisation. Frameworks like NVIDIA '
              'Triton Inference Server implement dynamic batching with configurable '
              'maximum latency constraints.'},

    # ── Faster data loading ───────────────────────────────────────────────────
    {'id': 6,  'topic': 'Data loading',     'cluster': 2,
     'title': 'DataLoader Workers and Prefetching',
     'text':  'Setting num_workers in PyTorch DataLoader to match available CPU cores '
              'enables parallel data preprocessing. Enabling pin_memory and prefetch_factor '
              'reduces data transfer overhead, cutting total training time by 20-40% for '
              'I/O-bound pipelines.'},

    {'id': 7,  'topic': 'Data loading',     'cluster': 2,
     'title': 'Dataset Caching with Memory-Mapped Files',
     'text':  'Pre-tokenising and caching datasets in Arrow format (HuggingFace datasets) '
              'or memory-mapped numpy arrays eliminates redundant preprocessing across '
              'training epochs. This is critical when tokenisation is the training '
              'bottleneck rather than GPU computation.'},

    # ── Faster convergence ────────────────────────────────────────────────────
    {'id': 8,  'topic': 'Convergence speed', 'cluster': 3,
     'title': 'Learning Rate Scheduling: Warmup and Cosine Decay',
     'text':  'Linear warmup followed by cosine annealing is the de facto schedule for '
              'transformer training. Warmup prevents early training instability, while '
              'cosine decay improves final model quality compared to step decay and '
              'reduces the number of epochs needed.'},

    {'id': 9,  'topic': 'Convergence speed', 'cluster': 3,
     'title': 'AdamW and Gradient Clipping for Stable Training',
     'text':  'AdamW decouples weight decay from the gradient update, improving '
              'generalisation over standard Adam. Combined with gradient clipping '
              '(max_norm=1.0), it prevents loss spikes and enables higher learning '
              'rates, leading to faster convergence.'},

    # ── Architecture efficiency ───────────────────────────────────────────────
    {'id': 10, 'topic': 'Architecture',     'cluster': 4,
     'title': 'FlashAttention for Memory-Efficient Transformers',
     'text':  'FlashAttention reimplements the attention mechanism to be IO-aware, '
              'reducing GPU memory bandwidth from O(N^2) to O(N). It enables training '
              'on longer sequences without out-of-memory errors and achieves 2-4x '
              'speedup over standard attention implementations.'},

    {'id': 11, 'topic': 'Architecture',     'cluster': 4,
     'title': 'Model Pruning: Structured and Unstructured Sparsity',
     'text':  'Magnitude-based pruning removes the smallest-weight parameters, creating '
              'sparse models that require fewer multiply-accumulate operations. Structured '
              'pruning removes entire attention heads or neurons, achieving speedup on '
              'standard hardware without sparse matrix libraries.'},
]

from collections import Counter
print(f'Corpus: {len(CORPUS)} docs across {len(set(d["topic"] for d in CORPUS))} topic clusters')
for topic, count in Counter(d['topic'] for d in CORPUS).items():
    ids = [d['id'] for d in CORPUS if d['topic'] == topic]
    print(f'  {topic:<22} {count} docs  ids={ids}')

In [ ]:
corpus_texts  = [d['text'] for d in CORPUS]
corpus_embeds = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=False)


def retrieve(query: str, top_k: int = 3):
    """Return top_k (doc, score) pairs by cosine similarity."""
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, corpus_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]
    return [(CORPUS[i], float(scores[i])) for i in indices]


print(f'Embedded {len(CORPUS)} docs. Retriever ready.')
print()

# Show the under-specification failure
print('=== Baseline: raw under-specified query ===')
print(f'Query: {original_query}')
print()
base_results = retrieve(original_query, top_k=5)
for doc, score in base_results:
    print(f'  [{doc["id"]}] [{doc["topic"]}] {doc["title"]} (score={score:.3f})')

topics_retrieved = set(doc['topic'] for doc, _ in base_results)
all_topics       = set(d['topic'] for d in CORPUS)
print()
print(f'Topics covered (top-5): {topics_retrieved}')
print(f'Topics MISSED:          {all_topics - topics_retrieved}')
print()
print('The single query lands in one neighbourhood, missing entire topic clusters.')

---
## 3. Multi-Query Fan-Out

Generate N alternative queries from different angles, retrieve for each, merge the results.

```
┌──────────────────────────────────────────────────────────────────────┐
│                 MULTI-QUERY FAN-OUT / FAN-IN                         │
└──────────────────────────────────────────────────────────────────────┘

              User query
  "How do I make my model faster?"
                 │
                 ▼  LLM rewriter
   ┌─────────────┬──────────────┬──────────────┬──────────────┐
   │             │              │              │              │
   ▼             ▼              ▼              ▼              ▼
 [Q1: train]  [Q2: infer]  [Q3: data]  [Q4: converge]  [Q5: arch]
   │             │              │              │              │
   ▼             ▼              ▼              ▼              ▼
 Retrieve     Retrieve      Retrieve      Retrieve       Retrieve
   │             │              │              │              │
   └─────────────┴──────────────┴──────────────┴──────────────┘
                                │
                         Deduplicate / MMR
                                │
                          Final context → LLM
```

In [ ]:
class MockMultiQueryLLM:
    """
    Simulates an LLM that generates N angle queries from different perspectives.

    In production, replace with a real LLM call (see Section 10).
    Key design: each angle query targets a DIFFERENT interpretation of the original.
    """

    _EXPANSIONS = {
        'model faster': [
            'What GPU training optimizations reduce time-per-epoch?',
            'What techniques reduce model inference latency at serving time?',
            'How do I eliminate data loading bottlenecks during model training?',
            'What optimizer and learning rate settings speed up model convergence?',
            'What model architecture modifications improve computational efficiency?',
        ],

        'training faster': [
            'What mixed precision and AMP techniques speed up GPU training throughput?',
            'How does gradient accumulation enable larger effective batch sizes?',
            'How do I parallelise training across multiple GPUs with DDP?',
            'What data pipeline optimizations reduce I/O bottlenecks in training?',
        ],

        'inference faster': [
            'How does post-training quantisation reduce inference latency?',
            'What is knowledge distillation and how does it produce smaller faster models?',
            'How does dynamic batching improve GPU utilisation at serving time?',
            'How does FlashAttention reduce memory and speed up transformer inference?',
        ],

        'training slow': [
            'What causes GPU training throughput bottlenecks in deep learning?',
            'How does I/O and data loading affect overall training time?',
            'What optimizer and schedule choices affect convergence speed?',
        ],

        'data loading': [
            'How do DataLoader workers and prefetching reduce training I/O time?',
            'What caching strategies eliminate repeated dataset preprocessing?',
        ],
    }

    def generate(self, query: str, n: int = 4) -> list:
        q = query.lower()
        for key, expansions in self._EXPANSIONS.items():
            if all(kw in q for kw in key.split()):
                return expansions[:n]
        # Generic fallback
        return [
            f'What are the theoretical foundations behind: {query}',
            f'What tools and frameworks implement: {query}',
            f'What are common mistakes when working on: {query}',
            f'What are practical benchmarks for: {query}',
        ][:n]


mq_llm = MockMultiQueryLLM()
print('MockMultiQueryLLM ready.')
print()
print(f'Angle queries for: {original_query}')
for i, q in enumerate(mq_llm.generate(original_query, n=5), 1):
    print(f'  Q{i}: {q}')

In [ ]:
def multi_retrieve(query: str, n_queries: int = 4, top_k_per_query: int = 3) -> tuple:
    """
    Fan-out: generate angle queries → retrieve for each → return raw merged list.

    Returns (raw_results, angle_queries) where raw_results may contain duplicates.
    """
    angle_qs = mq_llm.generate(query, n=n_queries)
    raw_results = []  # list of (doc, score, angle_query_idx)

    for idx, aq in enumerate(angle_qs):
        for doc, score in retrieve(aq, top_k=top_k_per_query):
            raw_results.append((doc, score, idx))

    return raw_results, angle_qs


# Demo: fan out and inspect the raw merged results
raw_results, angle_qs = multi_retrieve(original_query, n_queries=5, top_k_per_query=3)

print(f'Query: {original_query}')
print(f'Fan-out: {len(angle_qs)} angle queries x top-3 = {len(raw_results)} raw results')
print()
print(f'{"Angle Q":<6} {"Score":<8} {"Doc ID":<8} {"Topic":<22} {"Title"}')
print('-' * 90)
for doc, score, q_idx in raw_results:
    print(f'Q{q_idx+1:<5} {score:<8.3f} [{doc["id"]}]{" ":<5} {doc["topic"]:<22} {doc["title"]}')

# Count duplicates
doc_id_counts = Counter(doc['id'] for doc, _, _ in raw_results)
dups = {did: cnt for did, cnt in doc_id_counts.items() if cnt > 1}
print()
print(f'Unique docs: {len(doc_id_counts)}/{len(raw_results)} ({len(raw_results)-len(doc_id_counts)} duplicates)')
if dups:
    print(f'Duplicated doc IDs: {dups}')

---
## 4. Deduplication Strategies

Raw multi-query results contain duplicates — the same document retrieved by multiple angle queries. Two strategies:

| Strategy | How | Best for |
|---|---|---|
| **Simple set dedup** | Remove exact duplicates by ID; keep highest score | Fast; well-separated chunks |
| **MMR** | Remove near-duplicates by embedding similarity | Dense overlapping content; diversity matters |

In [ ]:
def simple_dedup(raw_results: list, keep: str = 'max') -> list:
    """
    Remove exact duplicates by doc ID. Keeps the version with the highest score.
    Returns list of (doc, score) sorted by score descending.
    """
    best = {}  # doc_id → (doc, score)
    for doc, score, *_ in raw_results:
        if doc['id'] not in best or score > best[doc['id']][1]:
            best[doc['id']] = (doc, score)
    return sorted(best.values(), key=lambda x: x[1], reverse=True)


deduped = simple_dedup(raw_results)

print(f'After simple deduplication: {len(deduped)} unique docs')
print()
print(f'{"Rank":<6} {"Score":<8} {"Doc ID":<8} {"Topic":<22} {"Title"}')
print('-' * 80)
for rank, (doc, score) in enumerate(deduped, 1):
    print(f'{rank:<6} {score:<8.3f} [{doc["id"]}]{" ":<5} {doc["topic"]:<22} {doc["title"]}')

topics_covered = set(doc['topic'] for doc, _ in deduped)
print()
print(f'Topics covered: {topics_covered}')
print(f'Topics missed:  {set(d["topic"] for d in CORPUS) - topics_covered}')

In [ ]:
def mmr_select(candidates: list, top_k: int = 6, lambda_param: float = 0.5) -> list:
    """
    Maximal Marginal Relevance selection.

    Scores each candidate as:
      MMR(d) = lambda * relevance(d) - (1 - lambda) * max_sim(d, already_selected)

    Args:
        candidates:    list of (doc, score) — output of simple_dedup()
        top_k:         number of documents to select
        lambda_param:  0 = maximum diversity, 1 = maximum relevance, 0.5 = balanced

    Returns list of (doc, score) in MMR selection order.
    """
    if not candidates:
        return []

    # Pre-compute embeddings for all candidates (one batch call)
    docs   = [doc for doc, _ in candidates]
    scores = np.array([s for _, s in candidates])
    embs   = embedder.encode(
        [d['text'] for d in docs],
        convert_to_tensor=False,
        show_progress_bar=False
    )  # shape (N, dim)

    # Normalise for cosine similarity via dot product
    norms       = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8
    embs_normed = embs / norms

    selected_idx   = []
    remaining_idx  = list(range(len(docs)))

    for _ in range(min(top_k, len(docs))):
        if not selected_idx:
            # First pick: highest relevance score
            best_i = int(np.argmax(scores[remaining_idx]))
        else:
            sel_embs = embs_normed[selected_idx]  # (n_selected, dim)
            best_mmr = -np.inf
            best_i   = None
            for i in remaining_idx:
                rel      = scores[i]
                max_sim  = float(np.max(embs_normed[i] @ sel_embs.T))
                mmr      = lambda_param * rel - (1 - lambda_param) * max_sim
                if mmr > best_mmr:
                    best_mmr = mmr
                    best_i   = i
            best_i = remaining_idx.index(best_i)

        chosen = remaining_idx.pop(best_i)
        selected_idx.append(chosen)

    return [(docs[i], float(scores[i])) for i in selected_idx]


mmr_results = mmr_select(deduped, top_k=6, lambda_param=0.5)

print('=== MMR selection (lambda=0.5, balanced relevance + diversity) ===')
print()
print(f'{"MMR rank":<10} {"Rel score":<12} {"Topic":<22} {"Title"}')
print('-' * 80)
for rank, (doc, score) in enumerate(mmr_results, 1):
    print(f'{rank:<10} {score:<12.3f} {doc["topic"]:<22} {doc["title"]}')

print()
print('MMR trades a small drop in average relevance score for greater topic diversity.')
print('The first pick is highest-relevance; each subsequent pick maximises the MMR score.')

In [ ]:
# Side-by-side comparison: baseline vs. simple dedup vs. MMR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

TOPIC_COLORS = {
    'Training speed':    '#1565C0',
    'Inference speed':   '#E53935',
    'Data loading':      '#2E7D32',
    'Convergence speed': '#F57F17',
    'Architecture':      '#6A1B9A',
}

def plot_results(ax, results, title, top_k=6):
    results = results[:top_k]
    topics   = [doc['topic'] for doc, _ in results]
    scores   = [s for _, s in results]
    labels   = [f'[{doc["id"]}]\n{doc["title"][:20]}...' for doc, _ in results]
    colors   = [TOPIC_COLORS.get(t, '#888') for t in topics]

    bars = ax.barh(range(len(results)), scores, color=colors, alpha=0.85)
    ax.set_yticks(range(len(results)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Score')
    ax.set_title(title, fontweight='bold')
    ax.invert_yaxis()

    topics_seen = set(topics)
    ax.text(0.98, 0.02, f'{len(topics_seen)} topics covered',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=9, style='italic',
            color='green' if len(topics_seen) >= 4 else 'red')

# Baseline: raw single query, top-6
plot_results(axes[0], retrieve(original_query, top_k=6),
             'Baseline: single query\n(top-6)')

# Simple dedup top-6
plot_results(axes[1], simple_dedup(raw_results)[:6],
             'Multi-query + simple dedup\n(top-6 by score)')

# MMR top-6
plot_results(axes[2], mmr_results[:6],
             'Multi-query + MMR\n(top-6 by relevance + diversity)')

# Shared legend
legend_handles = [mpatches.Patch(color=c, label=t) for t, c in TOPIC_COLORS.items()]
fig.legend(handles=legend_handles, loc='lower center', ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f'Query: {original_query}', style='italic', fontsize=11, y=1.02)
show_plot()

def count_topics(results):
    return len(set(doc['topic'] for doc, _ in results))

print(f'Topics covered (baseline top-6):      {count_topics(retrieve(original_query, top_k=6))}/5')
print(f'Topics covered (multi-query dedup):   {count_topics(simple_dedup(raw_results)[:6])}/5')
print(f'Topics covered (multi-query MMR):     {count_topics(mmr_results[:6])}/5')

---
## 5. Visualisation: How Angle Queries Cover Different Clusters

PCA projects from 384D to 2D to show which parts of embedding space each angle query targets.

In [ ]:
if not SKLEARN_AVAILABLE:
    print('sklearn not available — install with: pip install scikit-learn')
else:
    n_angle_qs  = 5
    angle_qs_5  = mq_llm.generate(original_query, n=n_angle_qs)

    # Embed corpus + original query + angle queries
    all_texts = corpus_texts + [original_query] + angle_qs_5
    all_embs  = embedder.encode(all_texts, convert_to_tensor=False, show_progress_bar=False)

    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(all_embs)

    doc_coords   = coords[:len(CORPUS)]
    orig_coord   = coords[len(CORPUS)]
    angle_coords = coords[len(CORPUS) + 1:]

    fig, ax = plt.subplots(figsize=(13, 9))

    # Plot corpus docs coloured by topic
    for doc, (x, y) in zip(CORPUS, doc_coords):
        color = TOPIC_COLORS.get(doc['topic'], '#888')
        ax.scatter(x, y, color=color, s=200, zorder=3, alpha=0.85, edgecolors='white', linewidths=0.8)
        ax.annotate(f'[{doc["id"]}]', (x, y), textcoords='offset points',
                    xytext=(5, 4), fontsize=8)

    # Plot original query
    ax.scatter(*orig_coord, color='black', marker='D', s=250, zorder=5,
               label=f'Original query', edgecolors='white')
    ax.annotate('Original\nquery', orig_coord, textcoords='offset points',
                xytext=(6, -14), fontsize=9, color='black', fontweight='bold')

    # Plot angle queries with connecting lines to original
    angle_markers = ['*', '^', 's', 'P', 'h']
    angle_colors  = ['#1565C0', '#E53935', '#2E7D32', '#F57F17', '#6A1B9A']
    angle_labels  = ['Q1: training', 'Q2: inference', 'Q3: data load', 'Q4: converge', 'Q5: arch']

    for i, (coord, aq) in enumerate(zip(angle_coords, angle_qs_5)):
        ax.scatter(*coord, color=angle_colors[i], marker=angle_markers[i],
                   s=350, zorder=5, label=f'{angle_labels[i]}: {aq[:35]}...')
        ax.annotate('', xy=coord, xytext=orig_coord,
                    arrowprops=dict(arrowstyle='->', color=angle_colors[i],
                                   lw=1.5, linestyle='dashed', alpha=0.7))

    # Legend for corpus topics
    for topic, color in TOPIC_COLORS.items():
        ax.scatter([], [], color=color, s=80, label=f'Corpus: {topic}')

    ax.set_title(
        'PCA (384D → 2D): Multi-query angle queries fan out to different corpus clusters\n'
        'Each angle query arrow points toward a different region — covering all topic clusters',
        fontweight='bold'
    )
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)')
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
    ax.grid(True, alpha=0.25)
    show_plot()

    print('Geometric intuition:')
    print('  The original query diamond lands at a compromise point.')
    print('  Each angle query arrow points toward a different topic cluster.')
    print('  Together, the angle queries cover the full corpus neighbourhood.')

---
## 6. When Multi-Query Hurts

Multi-query is powerful but not free. Three real costs:

In [ ]:
# Cost / latency breakdown
print('=== Latency Model: Multi-query vs. Baseline ===')
print()

rows = [
    ('Baseline (1 query)',          '0 ms',    '~200 ms',   '0 ms',   '~200 ms'),
    ('Multi-query (4 parallel)',    '~500 ms',  '~200 ms',   '~50 ms',  '~750 ms'),
    ('Multi-query (4 serial)',      '~500 ms',  '~800 ms',  '~50 ms',  '~1350 ms'),
]
print(f'{"Strategy":<30} {"LLM rewrite":<14} {"Retrieval":<12} {"Dedup":<10} {"Total"}')
print('-' * 80)
for row in rows:
    print(f'{row[0]:<30} {row[1]:<14} {row[2]:<12} {row[3]:<10} {row[4]}')

print()
print('Always run retrieval calls in parallel. The latency multiplier is then:')
print('  ~200ms baseline → ~750ms multi-query = 3.75x slower')
print()
print('For a streaming chatbot where the first token must arrive in < 1s,')
print('multi-query is often too slow. Use it for batch pipelines and async flows.')

In [ ]:
# Demonstrate query drift — when a bad rewrite pollutes the context
specific_query = 'What is the learning rate warmup schedule for training BERT?'

# A good multi-query expansion
good_angles = [
    'What learning rate warmup strategies are used for transformer pre-training?',
    'How does linear warmup improve early training stability in BERT?',
    'What is the recommended training schedule for large language model fine-tuning?',
]

# A bad expansion — drifted away from the original intent
drifted_angles = [
    'What is BERT and how does transformer architecture work?',
    'What are the best NLP models for text classification?',
    'How do I deploy a machine learning model to production?',  # way off
]

print(f'Specific query: {specific_query}')
print()

print('=== Good expansion (stays on topic) ===')
for i, q in enumerate(good_angles, 1):
    top = retrieve(q, top_k=1)
    print(f'  Q{i}: {q}')
    print(f'       → [{top[0][0]["id"]}] {top[0][0]["title"]} (score={top[0][1]:.3f})')

print()
print('=== Drifted expansion (pulls in irrelevant docs) ===')
for i, q in enumerate(drifted_angles, 1):
    top = retrieve(q, top_k=1)
    print(f'  Q{i}: {q}')
    print(f'       → [{top[0][0]["id"]}] {top[0][0]["title"]} (score={top[0][1]:.3f})')

print()
print('The third drifted query retrieves something unrelated — polluting the context.')
print()
print('Rules to avoid drift:')
print('  1. Use temperature=0 for the rewriter — creativity hurts here')
print('  2. Prompt the rewriter: "Stay close to the original intent. Do not generalise."')
print('  3. Add a relevance gate: discard angle queries with < 0.4 sim to original')

In [ ]:
# Relevance gate: filter angle queries that drift too far from the original
def relevance_gated_multi_retrieve(
    query: str, n_queries: int = 5,
    top_k_per_query: int = 3,
    gate_threshold: float = 0.6
) -> tuple:
    """
    Multi-retrieve with a relevance gate:
    discard any angle query whose cosine similarity to the original is below threshold.
    """
    angle_qs   = mq_llm.generate(query, n=n_queries)
    orig_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    angle_embs = embedder.encode(angle_qs, convert_to_tensor=True, show_progress_bar=False)

    sims       = util.cos_sim(orig_emb, angle_embs)[0].cpu().numpy()
    passed     = [(aq, float(s)) for aq, s in zip(angle_qs, sims) if s >= gate_threshold]
    filtered   = [(aq, float(s)) for aq, s in zip(angle_qs, sims) if s <  gate_threshold]

    raw_results = []
    for aq, sim in passed:
        for doc, score in retrieve(aq, top_k=top_k_per_query):
            raw_results.append((doc, score, aq))

    return raw_results, passed, filtered


raw, passed, filtered = relevance_gated_multi_retrieve(
    original_query, n_queries=5, gate_threshold=0.7
)

print(f'Original query: {original_query}')
print(f'Threshold: 0.70 similarity to original')
print()
print(f'Passed gate ({len(passed)} queries):')
for aq, sim in passed:
    print(f'  sim={sim:.3f}  {aq}')
print()
if filtered:
    print(f'Filtered out ({len(filtered)} queries — would have drifted):')
    for aq, sim in filtered:
        print(f'  sim={sim:.3f}  {aq}')
else:
    print(f'No queries filtered — all stayed close enough to the original.')

---
## 7. Framework Implementations

LangChain and LlamaIndex both ship multi-query retrieval out of the box. The code below is for reference — substitute your own vector store and LLM.

In [ ]:
# ── LangChain: MultiQueryRetriever ────────────────────────────────────────────
# pip install langchain langchain-openai langchain-community

langchain_code = '''
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Default: generates 3 alternative queries automatically
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Custom prompt: more control over the angle queries generated
custom_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Generate 5 versions of this question from different angles
(training speed, inference speed, data loading, convergence, architecture).
Stay close to the original intent.

Original: {question}

Versions (one per line):"""
)

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
    prompt=custom_prompt
)

# One call fans out to N queries and returns deduplicated union
docs = retriever.invoke("How do I make my model faster?")
'''

print('=== LangChain: MultiQueryRetriever ===')
print(langchain_code)
print('Under the hood: LangChain prompts the LLM for N alternative queries,')
print('retrieves for each, and returns the union (set deduplication by doc ID).')

In [ ]:
# ── LlamaIndex: SubQuestionQueryEngine ────────────────────────────────────────
# pip install llama-index llama-index-llms-openai

llamaindex_code = '''
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.llms.openai import OpenAI

# Wrap your index as a tool
ml_tool = QueryEngineTool(
    query_engine=index.as_query_engine(),
    metadata=ToolMetadata(
        name="ml_optimisation_docs",
        description="Documentation on ML model speed optimisation techniques"
    )
)

# SubQuestionQueryEngine decomposes the question and retrieves in parallel
engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[ml_tool],
    llm=OpenAI(model="gpt-4o-mini", temperature=0),
    use_async=True   # enables parallel sub-query execution
)

response = engine.query("How do I make my model faster?")
# LlamaIndex will: decompose → retrieve per sub-question → synthesise
print(response)
'''

print('=== LlamaIndex: SubQuestionQueryEngine ===')
print(llamaindex_code)
print('LlamaIndex goes further: it synthesises each sub-question independently,')
print('then combines the partial answers into a final response.')

---
## 8. The Agentic Connection

Multi-query is the first step toward agentic retrieval. The pattern is identical:

```
┌──────────────────────────────────────────────────────────────────────┐
│         MULTI-QUERY IS FAN-OUT / FAN-IN — THE AGENTIC SHAPE         │
└──────────────────────────────────────────────────────────────────────┘

  Multi-query RAG:                 Full Agentic RAG:

  User query                       User query
      │                                │
      ▼ LLM planner                    ▼ LLM planner
   [Q1][Q2][Q3][Q4]              [VectorDB][WebSearch][SQL][API]
      │   │   │   │                   │         │       │    │
      ▼   ▼   ▼   ▼              Retrieve   Fetch   Query  Call
      └───┴───┴───┘                   └─────────┴───────┴────┘
             │                                  │
         Fan-in                             Fan-in
        Dedup/MMR                       Merge + Rerank
             │                                  │
         LLM synth                          LLM synth
             │                                  │
          Answer                            Answer

  In multi-query:  the "actions" are all vector DB retrievals
  In a full agent: the actions are heterogeneous — each fan-out
                   goes to a DIFFERENT type of data source

  The fan-out / fan-in pattern is the same in both cases.
  Multi-query teaches you the mechanic; agents extend the action space.
```

In [ ]:
# Sketch the agentic extension
agentic_pipeline = {
    'query':   'How has mixed precision training adoption changed since 2020?',
    'actions': [
        {'type': 'vector_db',  'q': 'mixed precision training techniques AMP',
         'rationale': 'retrieve technical documentation on AMP'},
        {'type': 'web_search', 'q': 'mixed precision training adoption benchmark 2023 2024',
         'rationale': 'find recent adoption statistics'},
        {'type': 'sql',        'q': 'SELECT year, pct_models FROM training_survey WHERE technique="AMP"',
         'rationale': 'pull internal survey data'},
    ]
}

print('Agentic retrieval example:')
print(f'  Query: {agentic_pipeline["query"]}')
print()
print('  Actions (fan-out):')
for a in agentic_pipeline['actions']:
    print(f'    [{a["type"].upper():<12}] {a["q"]}')
    print(f'    Rationale: {a["rationale"]}')
    print()

print('Each action type requires different infrastructure (vector DB, search API, SQL).')
print('But the fan-out / fan-in orchestration is identical to multi-query.')
print()
print('This is why understanding multi-query is the prerequisite for agentic RAG.')
print('The agent just adds: (1) heterogeneous action types, (2) conditional fan-out,')
print('(3) iterative loops where result N determines action N+1.')

---
## 9. Coverage@k Evaluation

For under-specified queries with multiple relevant docs, the right metric is **Coverage@k** — what fraction of relevant topic clusters appear in the top-k results?

$$\text{Coverage@k} = \frac{|\text{retrieved topics} \cap \text{relevant topics}|}{|\text{relevant topics}|}$$

In [ ]:
EVAL_SET = [
    {
        'query':          'How do I make my model faster?',
        'relevant_topics': {'Training speed', 'Inference speed', 'Data loading',
                            'Convergence speed', 'Architecture'},
        'description':    'Under-specified — all 5 topic clusters are relevant',
    },
    {
        'query':          'How do I speed up model training?',
        'relevant_topics': {'Training speed', 'Data loading', 'Convergence speed'},
        'description':    'Training-focused — 3 clusters relevant',
    },
    {
        'query':          'How do I speed up model inference?',
        'relevant_topics': {'Inference speed', 'Architecture'},
        'description':    'Inference-focused — 2 clusters relevant',
    },
    {
        'query':          'How do I reduce data loading time during training?',
        'relevant_topics': {'Data loading'},
        'description':    'Specific single-cluster — multi-query should not hurt',
    },
]


def coverage_at_k(results: list, relevant_topics: set, k: int = 6) -> float:
    retrieved_topics = set(doc['topic'] for doc, _ in results[:k])
    return len(retrieved_topics & relevant_topics) / len(relevant_topics)


K = 6
print(f'Coverage@{K} Evaluation — fraction of relevant topic clusters found in top-{K}\n')
print(f'{"Query":<45} {"Baseline":<12} {"MQ+dedup":<12} {"MQ+MMR"}')
print('-' * 90)

base_covs  = []
dedup_covs = []
mmr_covs   = []

for item in EVAL_SET:
    q    = item['query']
    rel  = item['relevant_topics']

    base      = retrieve(q, top_k=K)
    raw_r, _  = multi_retrieve(q, n_queries=4, top_k_per_query=3)
    deduped_r = simple_dedup(raw_r)
    mmr_r     = mmr_select(deduped_r, top_k=K)

    b_cov  = coverage_at_k(base,     rel, K)
    d_cov  = coverage_at_k(deduped_r, rel, K)
    m_cov  = coverage_at_k(mmr_r,    rel, K)

    base_covs.append(b_cov)
    dedup_covs.append(d_cov)
    mmr_covs.append(m_cov)

    q_short = q[:42] + '...' if len(q) > 42 else q
    print(f'{q_short:<45} {b_cov:<12.0%} {d_cov:<12.0%} {m_cov:.0%}')

print()
print(f'{"Average":<45} {np.mean(base_covs):<12.0%} {np.mean(dedup_covs):<12.0%} {np.mean(mmr_covs):.0%}')
print()
print(f'Multi-query Coverage@{K} improvement: +{(np.mean(mmr_covs) - np.mean(base_covs)):.0%} absolute')

In [ ]:
# Visualise Coverage@k results
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x     = np.arange(len(EVAL_SET))
width = 0.26
q_labels = [item['query'][:28] + '...' for item in EVAL_SET]

# Left: per-query grouped bar chart
b1 = axes[0].bar(x - width, base_covs,  width, color='#E53935', alpha=0.8, label='Baseline')
b2 = axes[0].bar(x,         dedup_covs, width, color='#F57F17', alpha=0.8, label='MQ + simple dedup')
b3 = axes[0].bar(x + width, mmr_covs,   width, color='#2E7D32', alpha=0.8, label='MQ + MMR')

axes[0].set_xticks(x)
axes[0].set_xticklabels(q_labels, rotation=18, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.2)
axes[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
axes[0].set_ylabel(f'Coverage@{K} (fraction of relevant clusters found)')
axes[0].set_title(f'Coverage@{K} per Query', fontweight='bold')
axes[0].legend(fontsize=10)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.05:
            axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.02,
                         f'{h:.0%}', ha='center', fontsize=8)

# Right: average summary
strategies = ['Baseline\n(1 query)', 'Multi-query\n+ dedup', 'Multi-query\n+ MMR']
avgs       = [np.mean(base_covs), np.mean(dedup_covs), np.mean(mmr_covs)]
colors     = ['#E53935', '#F57F17', '#2E7D32']

bars = axes[1].bar(strategies, avgs, color=colors, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.15)
axes[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
axes[1].set_ylabel(f'Average Coverage@{K}')
axes[1].set_title(f'Average Coverage@{K} Across All Queries', fontweight='bold')

for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.0%}', ha='center', fontsize=13, fontweight='bold')

show_plot()

---
## 10. Using a Real LLM (Claude API)

In production, angle query generation goes to a real LLM. Use a small, fast model — query generation is a simple instruction-following task.

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass


MULTI_QUERY_PROMPT = """Generate {n} different versions of the following question,
each targeting a different interpretation or angle of the original.
Use specific, technical language that would appear in documentation.
Stay close to the original intent — do not generalise.
Output only the questions, one per line.

Original question: {query}

Versions:"""


if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_generate_angles(query: str, n: int = 5) -> list:
        """Use Claude to generate N angle queries."""
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',  # fast + cheap for this simple task
            max_tokens=512,
            messages=[{'role': 'user',
                       'content': MULTI_QUERY_PROMPT.format(n=n, query=query)}],
        )
        lines = response.content[0].text.strip().splitlines()
        return [l.strip().lstrip('0123456789.-) ') for l in lines if l.strip()][:n]

    def claude_multi_retrieve(query: str, n_queries: int = 5, top_k: int = 6) -> dict:
        """Full pipeline with Claude-generated angle queries."""
        angle_qs  = claude_generate_angles(query, n=n_queries)
        raw       = []
        for aq in angle_qs:
            for doc, score in retrieve(aq, top_k=3):
                raw.append((doc, score, aq))

        deduped = simple_dedup(raw)
        mmr     = mmr_select(deduped, top_k=top_k)

        return {'query': query, 'angle_queries': angle_qs,
                'raw': raw, 'deduped': deduped, 'mmr': mmr}

    # Demo
    print('=== Claude multi-query generation ===')
    result = claude_multi_retrieve(original_query, n_queries=5, top_k=6)
    print(f'Query: {result["query"]}')
    print()
    print('Angle queries (Claude-generated):')
    for i, aq in enumerate(result['angle_queries'], 1):
        print(f'  Q{i}: {aq}')
    print()
    print(f'MMR top-{len(result["mmr"])} results:')
    for doc, score in result['mmr']:
        print(f'  [{doc["id"]}] [{doc["topic"]}] {doc["title"]} ({score:.3f})')
    print()

    # Coverage comparison
    rel_all = set(d['topic'] for d in CORPUS)
    base_cov  = coverage_at_k(retrieve(original_query, top_k=6), rel_all, 6)
    mmr_cov   = coverage_at_k(result['mmr'], rel_all, 6)
    print(f'Coverage@6  Baseline: {base_cov:.0%}  Claude multi-query: {mmr_cov:.0%}')

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable this section:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('The MULTI_QUERY_PROMPT and claude_generate_angles() above show the full pattern.')
    print()
    print('Model sizing for angle query generation:')
    print('  claude-haiku-4-5-20251001  — recommended (fast, cheap, follows instructions well)')
    print('  claude-sonnet-4-6          — for complex domains needing nuanced interpretation')
    print()
    print('Always use temperature=0 for the rewriter.')
    print('You want deterministic, intent-preserving expansions, not creative variations.')

---
## Key Takeaways

1. **Under-specified queries are the root problem.** A single vague query embeds to a single compromise position in vector space. It retrieves from one neighbourhood and misses every other relevant cluster. Multi-query solves this by sending separate queries to each cluster.

2. **Multi-query = fan-out one question into N angle queries, retrieve for each, merge.** Each angle query is a different interpretation of the original. They embed to different regions, so together they cover far more of the relevant document space.

3. **Simple deduplication removes exact duplicates; MMR removes near-duplicates.** Use simple dedup when your chunks are well-separated. Use MMR when content is dense and overlapping — it selects documents that are relevant *and* diverse relative to what is already selected.

4. **Multi-query hurts when queries are already specific.** If the original query is precise and well-formed, angle queries introduce noise. A relevance gate (discard angle queries with similarity < threshold to original) is cheap insurance against drift.

5. **Always run retrieval calls in parallel.** Serial multi-query multiplies latency by N. Parallel cuts it back to ~single-query latency + LLM rewrite time.

6. **Multi-query is the fan-out / fan-in pattern — the shape of agentic retrieval.** In multi-query, all fan-out actions are vector DB lookups. In a full agent, actions are heterogeneous (vector DB, web search, SQL, APIs). Once you understand multi-query, you understand the orchestration mechanic of agentic RAG.

7. **Coverage@k is the right metric for under-specified queries.** Binary recall@k only checks if *one* relevant doc appears. Coverage@k checks how many relevant *topic clusters* appear — which is what actually matters when the user's question spans multiple distinct answers.

---

*Up next: Lesson 8.4 — Iterative retrieval: what happens when query #2 can only be formed after you have seen the answer to query #1? Multi-query fans out in parallel; iterative retrieval loops sequentially, using each result to decide what to retrieve next.*